Importando as bibliotecas e módulos necessários: 

In [23]:
import os
import pandas as pd
import numpy as np
import sys
import gc
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

Importando as bases necessárias: 

In [24]:
df_final = pd.read_parquet(r"C:\Users\emill\Downloads\TCC\processed\BASE_UNIF.parquet", engine="pyarrow")

In [25]:
df_final_2 = pd.read_parquet(r"C:\Users\emill\Downloads\TCC\processed\BASE_AED.parquet", engine="pyarrow")

Nessa etapa será feita a definição das variáveis a serem utilizadas:

• Dependente: nível de proficiência em matemática, categorizado;

• Independentes: a definir a partir da análise exploratória e de métricas de importância.

### 1. Categorização do nível de proficiência
Esta seção descreve o processo de categorização inicial a ser adotado: a classificação em 4 classes de proficiência (QEdu), e a divisão em 10 níveis (INEP), sendo eles:

• Insuficiente, para proficiências entre 0-224, Níveis 0 e 1

• Básico, para proficiências entre 225-299, Níveis 2, 3 e 4

• Proficiente, para proficiências entre 300-349, Níveis 5 e 6 

• Avançado, para proficiências a partir de 350, Níveis 7, 8 e 9

In [26]:
bins = [-float('inf'), 200, 225, 250, 275, 300, 325, 350, 375, 400, float('inf')]
labels_numericos = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
labels_proficiencia = [
    'Insuficiente', 
    'Básico',       
    'Proficiente',   
    'Avançado'     
]

### 2. Seleção de características baseada na Análise Exploratória

#### 2019
Colunas selecionadas: LOCALIZACAO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [27]:
df_2019 = df_final[df_final["ANO"] == 2019].copy()

df_2019['NIVEL_PROFICIENCIA'] = pd.cut(df_2019['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2019['PROFICIENCIA_DESCRICAO'] = df_2019['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'LOCALIZACAO', 'COR_RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC',     
    'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO'
]

df_2019 = df_2019[colunas]
df_2019.head()

,LOCALIZACAO,COR_RACA,ESCOL_MAE,FREQUENCIA_CONVERSA,QTD_COMPUTADOR,QTD_CARRO,GARAGEM,IDADE_INTROD_ESC,REPROVACAO,TEMPO_ESTUDO,TEMPO_TRAB_DOMES,POS_EF,PROFICIENCIA_DESCRICAO
0,1,Branca,Fundamental incompleto,Às vezes,Não respondeu,1 ou 2,Sim,Entre 4 e 7,Sim,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico
1,1,Parda,Superior completo,Sempre,0,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico
2,1,Parda,Não sabe,Às vezes,0,0,Não,Entre 4 e 7,Não,Entre 1 e 2 horas,Mais de 2 horas,Não sabe,Básico
3,1,Parda,Superior completo,Às vezes,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Entre 1 e 2 horas,Mais de 2 horas,Não sabe,Básico
4,1,Branca,Fundamental incompleto,Nunca,0,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Não respondeu,Estudar e trabalhar,Básico


In [28]:
# Verificando a distribuição das classes 
class_distribution = df_2019['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2019)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1042352 registros (54.52%)
Classe Insuficiente: 507526 registros (26.55%)
Classe Proficiente: 315123 registros (16.48%)
Classe Avançado: 46930 registros (2.45%)

Soma total de registros: 1911931 (Esperado: 1911931)


##### Análise de variância
Analisando variância - considerando que uma variável é potencialmente de baixa variabilidade quando mais de 95% dos registros estão em uma única categoria:

In [29]:
resultado = []

for coluna in df_2019.columns:

    if coluna == 'PROFICIENCIA_DESCRICAO':
        continue

    proporcao_max = (
        df_2019[coluna]
        .value_counts(normalize=True, dropna=False)
        .max()
    )

    resultado.append({
        'Variavel': coluna,
        'Categoria_dominante_%': proporcao_max * 100
    })

baixa_variabilidade = (
    pd.DataFrame(resultado)
    .sort_values(
        'Categoria_dominante_%',
        ascending=False
    )
)

print(baixa_variabilidade)

               Variavel  Categoria_dominante_%
0           LOCALIZACAO              89.246683
8            REPROVACAO              71.186460
11               POS_EF              59.526207
7      IDADE_INTROD_ESC              59.002809
6               GARAGEM              55.732241
3   FREQUENCIA_CONVERSA              49.008359
5             QTD_CARRO              48.682039
4        QTD_COMPUTADOR              47.041342
1              COR_RACA              44.154784
9          TEMPO_ESTUDO              41.100071
10     TEMPO_TRAB_DOMES              35.160212
2             ESCOL_MAE              27.021530


##### Análise de importância 
Analisando importância - 10 variáveis mais importantes dentre as selecionadas:

In [30]:
df_2019['PROFICIENCIA_DESCRICAO'] = (
    df_2019['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Avançado': 'Proficiente'
    })
)

y = df_2019['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

X = pd.get_dummies(
    df_2019.drop(columns=['PROFICIENCIA_DESCRICAO']),
    drop_first=True
)

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask]

X = X.astype(float)
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

arvore = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

arvore.fit(
    X_train,
    y_train
)

y_pred = arvore.predict(X_test)

importancias = pd.DataFrame({
    'Variavel': X.columns,
    'Importancia': arvore.feature_importances_
})

importancias = importancias.sort_values(
    'Importancia',
    ascending=False
)

print(importancias.head(10))

                             Variavel  Importancia
26                     REPROVACAO_Sim     0.348211
14              QTD_COMPUTADOR_1 ou 2     0.169012
15           QTD_COMPUTADOR_3 ou mais     0.067905
37             POS_EF_Somente estudar     0.067740
9                  ESCOL_MAE_Não sabe     0.054957
17                   QTD_CARRO_1 ou 2     0.048554
38           POS_EF_Somente trabalhar     0.040033
11  FREQUENCIA_CONVERSA_Não respondeu     0.034146
7            ESCOL_MAE_Médio completo     0.032694
5                      COR_RACA_Preta     0.026892


#### 2021
Colunas selecionadas: LOCALIZACAO, SEXO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [31]:
df_2021 = df_final_2[df_final_2["ANO"] == 2021].copy()

df_2021['NIVEL_PROFICIENCIA'] = pd.cut(df_2021['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2021['PROFICIENCIA_DESCRICAO'] = df_2021['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'LOCALIZACAO', 'SEXO', 'COR/RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC',     
    'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO'

]

df_2021 = df_2021[colunas]
df_2021.head()

,LOCALIZACAO,SEXO,COR/RACA,ESCOL_MAE,FREQUENCIA_CONVERSA,QTD_COMPUTADOR,QTD_CARRO,GARAGEM,IDADE_INTROD_ESC,REPROVACAO,TEMPO_ESTUDO,TEMPO_TRAB_DOMES,POS_EF,PROFICIENCIA_DESCRICAO
0,1,Masculino,Branca,Médio completo,Às vezes,0,0,Sim,3 ou menos,Não,Entre 1 e 2 horas,Mais de 2 horas,Estudar e trabalhar,Básico
1,1,Masculino,Parda,Não sabe,Sempre,1 ou 2,0,Não,Entre 4 e 7,Sim,Menos de 1 hora,Mais de 2 horas,Somente trabalhar,Insuficiente
2,1,Feminino,Branca,Fundamental incompleto,Às vezes,1 ou 2,Não respondeu,Sim,Entre 4 e 7,Não respondeu,Menos de 1 hora,Não usa o tempo para isso,Estudar e trabalhar,Insuficiente
3,1,Feminino,Parda,Não sabe,Nunca,0,1 ou 2,Não,Entre 4 e 7,Sim,Entre 1 e 2 horas,Menos de 1 hora,Estudar e trabalhar,Insuficiente
4,1,Feminino,Outra,Não sabe,Às vezes,0,0,Não,Entre 4 e 7,Não,Menos de 1 hora,Entre 1 e 2 horas,Estudar e trabalhar,Básico


In [32]:
# Verificando a distribuição das classes 
class_distribution = df_2021['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2021)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1016280 registros (53.88%)
Classe Insuficiente: 569072 registros (30.17%)
Classe Proficiente: 270061 registros (14.32%)
Classe Avançado: 30766 registros (1.63%)

Soma total de registros: 1886179 (Esperado: 1886179)


##### Análise de variância 
Analisando variância - considerando que uma variável é potencialmente de baixa variabilidade quando mais de 95% dos registros estão em uma única categoria:

In [33]:
resultado = []

for coluna in df_2021.columns:

    if coluna == 'PROFICIENCIA_DESCRICAO':
        continue

    proporcao_max = (
        df_2021[coluna]
        .value_counts(normalize=True, dropna=False)
        .max()
    )

    resultado.append({
        'Variavel': coluna,
        'Categoria_dominante_%': proporcao_max * 100
    })

baixa_variabilidade = (
    pd.DataFrame(resultado)
    .sort_values(
        'Categoria_dominante_%',
        ascending=False
    )
)

print(baixa_variabilidade)

               Variavel  Categoria_dominante_%
0           LOCALIZACAO              88.772168
9            REPROVACAO              74.366484
7               GARAGEM              58.190235
8      IDADE_INTROD_ESC              57.769915
12               POS_EF              56.213647
6             QTD_CARRO              48.721039
1                  SEXO              48.588973
5        QTD_COMPUTADOR              43.397790
2              COR/RACA              43.378174
4   FREQUENCIA_CONVERSA              42.870958
10         TEMPO_ESTUDO              40.457984
11     TEMPO_TRAB_DOMES              30.623074
3             ESCOL_MAE              30.211767


##### Análise de importância 
Analisando importância - 10 variáveis mais importantes dentre as selecionadas:

In [34]:
df_2021['PROFICIENCIA_DESCRICAO'] = (
    df_2021['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Avançado': 'Proficiente'
    })
)

y = df_2021['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

X = pd.get_dummies(
    df_2021.drop(columns=['PROFICIENCIA_DESCRICAO']),
    drop_first=True
)

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask]

X = X.astype(float)
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

arvore = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

arvore.fit(
    X_train,
    y_train
)

y_pred = arvore.predict(X_test)

importancias = pd.DataFrame({
    'Variavel': X.columns,
    'Importancia': arvore.feature_importances_
})

importancias = importancias.sort_values(
    'Importancia',
    ascending=False
)

print(importancias.head(10))

                       Variavel  Importancia
29               REPROVACAO_Sim     0.284005
17        QTD_COMPUTADOR_1 ou 2     0.158567
1                SEXO_Masculino     0.085511
18     QTD_COMPUTADOR_3 ou mais     0.079488
12           ESCOL_MAE_Não sabe     0.065711
41     POS_EF_Somente trabalhar     0.056093
20             QTD_CARRO_1 ou 2     0.046202
8                COR/RACA_Preta     0.034081
13  ESCOL_MAE_Superior completo     0.032383
10     ESCOL_MAE_Médio completo     0.026146


#### 2023
Colunas selecionadas: LOCALIZACAO, SEXO, COR/RACA, ESCOL_MAE, FREQUENCIA_CONVERSA, QTD_COMPUTADOR, QTD_CARRO, GARAGEM, IDADE_INTROD_ESC, REPROVACAO, TEMPO_ESTUDO, TEMPO_TRAB_DOMES, POS_EF

In [35]:
df_2023 = df_final_2[df_final_2["ANO"] == 2023].copy()

df_2023['NIVEL_PROFICIENCIA'] = pd.cut(df_2023['PROFICIENCIA_SAEB'], bins=bins, labels=labels_numericos, right=False)

nivel_to_proficiencia = {0: 'Insuficiente', 1: 'Insuficiente', 2: 'Básico', 3: 'Básico', 4: 'Básico',
                         5: 'Proficiente', 6: 'Proficiente', 7: 'Avançado', 8: 'Avançado', 9: 'Avançado'}

df_2023['PROFICIENCIA_DESCRICAO'] = df_2023['NIVEL_PROFICIENCIA'].map(nivel_to_proficiencia)

colunas = [
    'LOCALIZACAO', 'SEXO', 'COR/RACA', 'ESCOL_MAE', 'FREQUENCIA_CONVERSA', 'QTD_COMPUTADOR', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC',     
    'REPROVACAO', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES', 'POS_EF', 'PROFICIENCIA_DESCRICAO'

]

df_2023 = df_2023[colunas]
df_2023.head()

,LOCALIZACAO,SEXO,COR/RACA,ESCOL_MAE,FREQUENCIA_CONVERSA,QTD_COMPUTADOR,QTD_CARRO,GARAGEM,IDADE_INTROD_ESC,REPROVACAO,TEMPO_ESTUDO,TEMPO_TRAB_DOMES,POS_EF,PROFICIENCIA_DESCRICAO
1886179,1,Masculino,Parda,Médio completo,Às vezes,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Não usa o tempo para isso,Mais de 2 horas,Somente trabalhar,Insuficiente
1886180,1,Feminino,Preta,Não sabe,Não respondeu,Não respondeu,Não respondeu,Não respondeu,Entre 4 e 7,Não,Não respondeu,Não respondeu,Somente estudar,Insuficiente
1886181,1,Feminino,Parda,Fundamental incompleto,Não respondeu,0,1 ou 2,Não,Entre 4 e 7,Sim,Menos de 1 hora,Mais de 2 horas,Somente estudar,Básico
1886182,1,Masculino,Não respondeu,Fundamental incompleto,Nunca,1 ou 2,1 ou 2,Não,Entre 4 e 7,Sim,Menos de 1 hora,Entre 1 e 2 horas,Estudar e trabalhar,Básico
1886183,1,Feminino,Parda,Médio completo,Sempre,1 ou 2,1 ou 2,Sim,Entre 4 e 7,Não,Menos de 1 hora,Mais de 2 horas,Estudar e trabalhar,Básico


In [36]:
# Verificando a distribuição das classes 
class_distribution = df_2023['PROFICIENCIA_DESCRICAO'].value_counts()

# Total de registros no dataframe
total_records = len(df_2023)

print("Distribuição de registros:")
for classe, quantidade in class_distribution.items():
    porcentagem = (quantidade / total_records) * 100
    print(f"Classe {classe}: {quantidade} registros ({porcentagem:.2f}%)")

# Verificando se a soma total bate
print(f"\nSoma total de registros: {class_distribution.sum()} (Esperado: {total_records})")


Distribuição de registros:
Classe Básico: 1052094 registros (51.26%)
Classe Insuficiente: 655152 registros (31.92%)
Classe Proficiente: 294108 registros (14.33%)
Classe Avançado: 51302 registros (2.50%)

Soma total de registros: 2052656 (Esperado: 2052656)


##### Análise de variância 
Analisando variância - considerando que uma variável é potencialmente de baixa variabilidade quando mais de 95% dos registros estão em uma única categoria:

In [37]:
resultado = []

for coluna in df_2023.columns:

    if coluna == 'PROFICIENCIA_DESCRICAO':
        continue

    proporcao_max = (
        df_2023[coluna]
        .value_counts(normalize=True, dropna=False)
        .max()
    )

    resultado.append({
        'Variavel': coluna,
        'Categoria_dominante_%': proporcao_max * 100
    })

baixa_variabilidade = (
    pd.DataFrame(resultado)
    .sort_values(
        'Categoria_dominante_%',
        ascending=False
    )
)

print(baixa_variabilidade)

               Variavel  Categoria_dominante_%
0           LOCALIZACAO              89.093643
9            REPROVACAO              75.586167
7               GARAGEM              57.019491
12               POS_EF              56.804842
8      IDADE_INTROD_ESC              51.696631
1                  SEXO              48.959348
6             QTD_CARRO              47.711112
2              COR/RACA              46.628807
5        QTD_COMPUTADOR              45.930200
10         TEMPO_ESTUDO              42.838011
4   FREQUENCIA_CONVERSA              42.769222
11     TEMPO_TRAB_DOMES              33.052153
3             ESCOL_MAE              28.333194


##### Análise de importância 
Analisando importância - 10 variáveis mais importantes dentre as selecionadas:

In [38]:
df_2023['PROFICIENCIA_DESCRICAO'] = (
    df_2023['PROFICIENCIA_DESCRICAO']
    .replace({
        'Básico': 'Insuficiente',
        'Avançado': 'Proficiente'
    })
)

y = df_2023['PROFICIENCIA_DESCRICAO'].map({
    'Insuficiente': 0,
    'Proficiente': 1
})

X = pd.get_dummies(
    df_2023.drop(columns=['PROFICIENCIA_DESCRICAO']),
    drop_first=True
)

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask]

X = X.astype(float)
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

arvore = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

arvore.fit(
    X_train,
    y_train
)

y_pred = arvore.predict(X_test)

importancias = pd.DataFrame({
    'Variavel': X.columns,
    'Importancia': arvore.feature_importances_
})

importancias = importancias.sort_values(
    'Importancia',
    ascending=False
)

print(importancias.head(10))

                       Variavel  Importancia
29               REPROVACAO_Sim     0.250683
17        QTD_COMPUTADOR_1 ou 2     0.144444
18     QTD_COMPUTADOR_3 ou mais     0.087974
1                SEXO_Masculino     0.084361
41     POS_EF_Somente trabalhar     0.064968
12           ESCOL_MAE_Não sabe     0.048433
8                COR/RACA_Preta     0.039710
10     ESCOL_MAE_Médio completo     0.037285
13  ESCOL_MAE_Superior completo     0.033916
40       POS_EF_Somente estudar     0.033558


### 3. Persistência

Será salvo um parquet para cada ano apenas com as colunas finais. Os notebooks de modelagem seguintes carregam daqui — não repetem todo o pipeline.

In [39]:
#Exclusão das colunas que não ficaram entre as 10 mais importantes em cada ano
df_2019.drop(columns=['LOCALIZACAO', 'GARAGEM', 'IDADE_INTROD_ESC', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES'])
df_2021.drop(columns=['LOCALIZACAO', 'FREQUENCIA_CONVERSA', 'GARAGEM', 'IDADE_INTROD_ESC', 'TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES'])
df_2023.drop(columns=['LOCALIZACAO', 'FREQUENCIA_CONVERSA', 'QTD_CARRO', 'GARAGEM', 'IDADE_INTROD_ESC','TEMPO_ESTUDO', 'TEMPO_TRAB_DOMES'])

processed_dir = r"C:\Users\emill\Downloads\TCC\processed"
os.makedirs(processed_dir, exist_ok=True)

out_path = os.path.join(processed_dir, "Features_2019.parquet")
df_2019 = pd.DataFrame(df_2019)
df_2019.reset_index(drop=True, inplace=True)
df_2019.to_parquet(out_path, index=False, engine="pyarrow")

out_path = os.path.join(processed_dir, "Features_2021.parquet")
df_2021 = pd.DataFrame(df_2021)
df_2021.reset_index(drop=True, inplace=True)
df_2021.to_parquet(out_path, index=False, engine="pyarrow")

out_path = os.path.join(processed_dir, "Features_2023.parquet")
df_2023 = pd.DataFrame(df_2023)
df_2023.reset_index(drop=True, inplace=True)
df_2023.to_parquet(out_path, index=False, engine="pyarrow")